# 🚙 교통·에너지 탐구 도구 — Colab 시작하기

**설치할 것이 없다.** 브라우저만 있으면 된다.

사용법: 각 칸(셀) 왼쪽의 **▶ 재생 버튼**을 위에서부터 차례로 누른다.
처음 두 칸은 한 번만 누르면 되고, 그 아래는 몇 번이든 다시 눌러도 된다.

> ⚠️ Colab은 한동안 안 쓰면 초기화된다. 그러면 1번·2번 칸을 다시 누르면 된다.


## 1단계 — 준비 (한 번만)
도구를 내려받고 한글 폰트를 설치한다. 1~2분 걸린다.

In [ ]:
!git clone -q https://github.com/waterfirst/simulation.git 2>/dev/null || echo "이미 받음"
!pip -q install numpy scipy matplotlib
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm, matplotlib, shutil, os
shutil.rmtree(matplotlib.get_cachedir(), ignore_errors=True)
fm._load_fontmanager(try_read_cache=False)
matplotlib.rcParams['font.family'] = ['DejaVu Sans', 'NanumGothic']
matplotlib.rcParams['axes.unicode_minus'] = False
os.chdir('/content/simulation/eco-drive-kit/tools')
print('준비 끝. 현재 폴더:', os.getcwd())
print('도구 목록:', sorted(f for f in os.listdir() if f.endswith('.py')))

## 2단계 — 그림이 화면에 보이게 하는 도우미
아래 칸을 누르면 `run()` 이라는 명령이 생긴다. 앞으로 이걸로 도구를 돌린다.

In [ ]:
import subprocess, sys, os
from IPython.display import Image, display

def run(cmd, show=None):
    """도구를 돌리고, 만들어진 그림을 바로 보여준다."""
    print('$', cmd, '\n' + '-'*60)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = '\n'.join(l for l in r.stdout.splitlines() if 'Warning' not in l and 'warn(' not in l)
    print(out)
    if r.returncode != 0:
        print('[오류]', r.stderr.splitlines()[-1] if r.stderr else '')
    if show and os.path.exists(show):
        display(Image(show))
    return r

print('준비 끝. 이제 아래 칸들을 눌러보면 된다.')

---
# 🚗 가은·지효 — 자동차 효율

## 운전 습관 분석 (drive_score.py)
같은 길을 **거칠게** 몬 경우와 **부드럽게** 몬 경우를 비교한다.

In [ ]:
run('python drive_score.py ../data/demo_rough.csv --limit 60 --plot /content/a.png', '/content/a.png')

In [ ]:
run('python drive_score.py ../data/demo_smooth.csv --limit 60 --plot /content/b.png', '/content/b.png')

### 🔎 직접 해볼 것
- `--limit 60` 을 `--limit 50` 으로 바꾸면 과속 횟수가 어떻게 되나?
- `--mass 1400` 을 `--mass 2000`(SUV)으로 바꾸면 버린 에너지가 얼마나 늘어나나?

**너희가 찍은 데이터를 쓰려면**: 왼쪽 📁 아이콘 → 파일 업로드 → 아래처럼 경로만 바꾼다.
```
run('python drive_score.py /content/내파일.csv --limit 50 --speed-unit ms --plot /content/c.png', '/content/c.png')
```
`--speed-unit ms` 를 빠뜨리면 안 된다 (phyphox는 m/s로 준다).

## 급가속 제한 알고리즘 (accel_limiter.py)
탐지가 아니라 **제어**다. 네 가지 제어기를 같은 경로로 돌려 비교한다.

In [ ]:
run('python accel_limiter.py --plot /content/lim.png', '/content/lim.png')

### 🔎 직접 해볼 것
- `--stops 8 --seg 300 --vtgt 60` (더 빽빽한 도심) 에서는 어떤 제어기가 유리한가?
- `--stops 3 --seg 1000 --vtgt 70` (간선도로) 에서는?

**눈여겨볼 것**: 연료는 거의 안 바뀌는데 유해배출만 절반으로 준다. **왜 그럴까?**

In [ ]:
run('python accel_limiter.py --stops 8 --seg 300 --vtgt 60')

---
# 🎨 시아 — 공기저항·디자인

## 가상 풍동 (windtunnel.py)
차 모양에 따라 공기 흐름과 항력이 어떻게 달라지는지 본다.
**4~6분 걸린다.** 누르고 기다려라.

In [ ]:
run('python windtunnel.py --compare --steps 3000 --plot /content/wt.png', '/content/wt.png')

## 공기저항 역산 (coastdown.py)
자전거나 차가 관성으로 굴러가며 느려지는 모양에서 **공기저항계수 Cd·A** 를 뽑는다.
먼저 데모(합성 데이터)로 제대로 복원되는지 확인한다.

In [ ]:
run('python coastdown.py --demo --mass 85 --plot /content/cd.png', '/content/cd.png')

> 정답 CdA 0.450 을 0.46 정도로 복원하면 성공이다.
> **너희 데이터로**: `python coastdown.py /content/내측정.csv --mass 75 --unit ms --plot /content/cd2.png`
> `--mass` 는 **사람+자전거 전체 무게**다. 틀리면 결과가 통째로 틀린다.

---
# 🍃 주아 — 에너지 자립

## 어디까지 자급자족이 되나 (energy_budget.py)
태양광·풍력·압전을 **같은 단위(1㎡당 평균 W)** 로 세우고,
교통 인프라 부하와 비교한다.

In [ ]:
run('python energy_budget.py --plot /content/en.png', '/content/en.png')

### 🔎 직접 해볼 것
- `--solar-area 4` 를 붙이면 패널 4㎡로 무엇을 몇 개 돌릴 수 있는지 나온다
- `--wind-speed 5` 로 바꾸면 풍력이 얼마나 좋아지나? (전력은 풍속의 **세제곱**에 비례)

**눈여겨볼 것**: 압전 막대가 안 보여서 세로축을 로그로 그렸다. 그 자체가 결론이다.

In [ ]:
run('python energy_budget.py --solar-area 4 --wind-speed 5')

---
# 🏙️ 인프라·시스템

## 신호 연동 시뮬 (glosa.py)
개인의 운전 습관보다 **신호 시스템**이 얼마나 더 줄이는지 본다.

In [ ]:
run('python glosa.py --signals 8 --plot /content/gl.png', '/content/gl.png')

### 🔎 직접 해볼 것
- `--cycle 60` (짧은 주기) 로 바꾸면 GLOSA 효과가 커지나?
- `--spacing 800` (넓은 블록) 이면?

## 우리 동네 교차로 조사 (survey.py)
**직접 잰 신호 주기**를 넣으면 지역 단위 효과가 나온다. 먼저 양식을 만든다.

In [ ]:
run('python survey.py --template /content/우리동네.csv')
print('\n왼쪽 📁 에서 /content/우리동네.csv 를 받아 엑셀로 채운 뒤 다시 업로드해라.')

In [ ]:
run('python survey.py ../data/survey_demo.csv --plot /content/sv.png', '/content/sv.png')

---
# ⚖️ 지윤 — 법제화

## 도로 영향평가(안) (road_impact.py)
공공도로를 없앨 때의 영향을 **점수로** 만들고, 우회 CO2 를 계산한다.

In [ ]:
run('python road_impact.py --template /content/도로평가.csv')

In [ ]:
run('python road_impact.py ../data/road_example.csv --plot /content/rd.png', '/content/rd.png')

### 🔎 직접 해볼 것
`road_impact.py` 안의 `WEIGHTS`(배점)를 바꾸면 총점이 달라진다.
**기준선 50점을 넘나드나?** 넘나든다면 그게 이 평가표의 한계이고,
"배점을 공청회에서 합의해야 한다"는 주장의 근거가 된다.

---
## 📌 마무리 — 그림 저장하기
왼쪽 **📁 파일** 아이콘을 누르면 `/content/` 안에 만들어진 그림들이 보인다.
점 세 개(⋮) → **다운로드** 로 받아서 보고서에 넣어라.

**가장 중요한 것**: 이 도구들을 그냥 돌린 결과는 너희 탐구가 아니다.
**숫자를 바꿔보고, 왜 그렇게 되는지 설명한 부분**이 너희 것이다.